# Simulating the data-generating process

Erik, 2026-11-18

A statistical model is a story about how the data came to be. Writing that
story as code that simulates fake data tests the model before any real
observations are fit. Simulation makes assumptions concrete: when fake data
look nothing like real data, an assumption is worth questioning. Because the
true parameters are known, simulated data also show whether a fitting
procedure can recover those values and how precisely a sample of a given size
estimates them.

This notebook builds a simulation for the regression in `02-ols-vs-bayes`,
compares fake students with the HSLS:09 sample, checks parameter recovery in
Stan, and runs simulation-based calibration to verify that the posterior
intervals have the coverage they claim.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

fw.scaffold()
import bayes  # functions/bayes.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")

clean = fw.data_read("inputs.final.hsls_clean", dtype={"student_id": str})
hsls = bayes.analysis_sample(clean)
regression = bayes.model("regression")

# One generator for every simulation, so reruns give the same fake data
rng = np.random.default_rng(1009)

## The story as code

The regression models each student's follow-up math score conditional on two
predictors:

$$
\text{w2\_math\_z}_i \sim \text{Normal}(\alpha + \beta_1\, \text{w1\_math\_z}_i + \beta_2\, \text{w1\_ses}_i,\ \sigma).
$$

Both math scores are in SD units, the T-score minus 50, divided by 10, as
`02-ols-vs-bayes` explains.

That equation says nothing about where the predictors come from. Simulating
complete student records requires a model for them as well, so the code draws
base-year math and SES from a bivariate normal distribution matching the
HSLS:09 means, standard deviations, and correlation before generating the
follow-up score. The function below implements this full process and is
identical to `bayes.simulate()`.

In [ ]:
def simulate_students(n, alpha, beta, sigma, rng):
    # Step 1: predictors, in SD units. Base-year math has mean 0.17 and SD 1
    # (51.7 and 10 as a T-score), SES has mean 0.08 and SD 0.8, and the two
    # correlate 0.43.
    mean = np.array([0.17, 0.08])
    sd = np.array([1.0, 0.8])
    covariance = np.array([[1.0, 0.43], [0.43, 1.0]]) * np.outer(sd, sd)
    x = rng.multivariate_normal(mean, covariance, size=n)

    # Step 2: each student's expected follow-up score
    mu = alpha + x @ np.asarray(beta)

    # Step 3: the follow-up score, with normal noise around the expectation
    y = rng.normal(mu, sigma)

    # Both math scores in SD units and as T-scores
    fake = pd.DataFrame({"w1_math_z": x[:, 0], "w1_ses": x[:, 1], "w2_math_z": y})
    fake["w1_math"] = 50 + 10 * fake["w1_math_z"]
    fake["w2_math"] = 50 + 10 * fake["w2_math_z"]
    return fake


# The HSLS:09 values behind the simulation's settings
pd.Series(
    {
        "Mean of w1_math_z": hsls["w1_math_z"].mean(),
        "SD of w1_math_z": hsls["w1_math_z"].std(),
        "Mean of w1_ses": hsls["w1_ses"].mean(),
        "SD of w1_ses": hsls["w1_ses"].std(),
        "Correlation": hsls["w1_math_z"].corr(hsls["w1_ses"]),
    }
).round(2).to_frame("HSLS:09")

The simulation settings round these HSLS:09 values. Base-year math in this
sample averages 0.17 SD above the T-score mean of 50 because the sample includes
only students with both scores, who scored higher than students missing one.

## Fake students next to real ones

Using parameter values close to the HSLS:09 estimates, the code simulates a
cohort of the same size as the HSLS:09 sample and plots the distribution of
each variable alongside the observed data to compare their centers, spreads,
and shapes. The plots and the table show both math scores as T-scores, the
scale the test reports.

In [ ]:
truth = {"alpha": 0.04, "beta": [0.7, 0.18], "sigma": 0.66}
fake = simulate_students(len(hsls), truth["alpha"], truth["beta"], truth["sigma"], rng)

both = pd.concat([hsls.assign(Source="HSLS:09"), fake.assign(Source="Simulated")], ignore_index=True)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for name, ax in zip(["w1_math", "w1_ses", "w2_math"], axes):
    sns.kdeplot(both, x=name, hue="Source", common_norm=False, ax=ax)
    ax.set_xlabel(f"{name} (T-score)" if name.endswith("math") else name)
    ax.set_ylabel("Density")
fig.suptitle(f"Simulated and HSLS:09 students (n = {len(hsls):,} each)")
fig.tight_layout()

In [ ]:
summary = both.groupby("Source")[["w1_math", "w1_ses", "w2_math"]].agg(["mean", "std", "min", "max"]).round(2)
summary.T

The simulated distributions match the observed means and standard deviations
because the predictor settings come from the HSLS:09 summaries and the outcome
parameters come from the HSLS:09 regression fit. Where the plots diverge is in
their shapes. In HSLS:09, the SES composite is bounded and lumpy because it
combines categorical parent education and occupation with bracketed income,
whereas the simulated normal draws are smooth and unbounded. Observed math
T-scores also stop at lower and upper limits set by the test, while normal
noise pushes simulated scores past both bounds.

These two discrepancies have different implications for the regression.
Because the model conditions on the predictors, the irregular distribution of
SES does not bias the regression estimates. The outcome distribution, however,
is part of the model itself, and the posterior predictive checks in
`08-posterior-predictive` look directly for those score bounds.

## Recovering known parameters

A fitting procedure that cannot recover known parameters from data generated
by its own model will not recover them from real data. Using the weak priors
from `bayes.WEAK`, the code fits the Stan model to 1,000 simulated students
and summarizes the posterior estimates and 90% intervals against the true
parameter values.

In [ ]:
fake_small = simulate_students(1000, truth["alpha"], truth["beta"], truth["sigma"], rng)
fit = bayes.sample(regression, bayes.stan_data(fake_small, bayes.WEAK))
draws = bayes.draws(fit)

truth_values = {"alpha": truth["alpha"], "beta[1]": truth["beta"][0], "beta[2]": truth["beta"][1], "sigma": truth["sigma"]}
rows = {}
for name, value in truth_values.items():
    low, high = bayes.interval(draws[name], 0.9)
    rows[name] = {"Truth": value, "Posterior mean": draws[name].mean(), "90% low": low, "90% high": high, "Truth inside": low <= value <= high}
pd.DataFrame.from_dict(rows, orient="index").round(3)

The last column of the table indicates whether each 90% posterior interval
contains the true value. A miss in a single simulation is not evidence of a
problem. Bayesian intervals are calibrated on average over parameter values
drawn from the prior, whereas at one fixed set of true values, the miss rate
across repeated data sets can depart from 10%. Checking that prior-averaged
rate requires repeating the fit across many parameter draws, as the
calibration section below does over 200 fits.

## Precision and sample size

Simulation also shows how posterior uncertainty shrinks as the sample grows.
Fitting the model to simulated samples ranging from 50 to 20,000 students, the
code plots the posterior standard deviation of the SES slope against sample
size on logarithmic axes. Because regression uncertainty falls with the square
root of the sample size, the plotted points lie close to a straight line with
slope −1/2.

In [ ]:
rows = []
for n in [50, 200, 800, 3200, 12800, 20000]:
    sample = simulate_students(n, truth["alpha"], truth["beta"], truth["sigma"], rng)
    draws_n = bayes.draws(bayes.sample(regression, bayes.stan_data(sample, bayes.WEAK)))
    rows.append({"n": n, "Posterior SD of the SES slope": draws_n["beta[2]"].std()})
precision = pd.DataFrame(rows)

slope = np.polyfit(np.log(precision["n"]), np.log(precision["Posterior SD of the SES slope"]), 1)[0]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(precision["n"], precision["Posterior SD of the SES slope"], "o-")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Simulated students (log scale)")
ax.set_ylabel("Posterior SD of the SES slope (log scale)")
ax.set_title(f"Posterior SD by sample size (fitted log-log slope {slope:.2f})")
plt.tight_layout()
precision.round(3)

## Simulation-based calibration

A single recovery check can pass by luck. Simulation-based calibration (Talts
et al., 2018) tests recovery systematically across the prior distribution.
Each replication draws a set of true parameters from the prior, simulates a
data set from the model under those values, fits the model to the simulated
data, and records each true parameter's rank: the count of posterior draws
that fall below it.

If the model and sampler are correct, each true value behaves like a draw from
its own posterior, making every rank equally likely. Across many replications,
a histogram of those ranks should be flat. Systematic departures diagnose
specific problems: a U-shaped histogram means the posteriors are too narrow, a
central hump means they are too wide, and a slope means they are shifted.

The code runs 200 simulations of 100 students each, drawing 100 HSLS:09
students at random on every run and using their observed predictors so the
predictor distribution remains realistic. Each fit retains 99 of its 4,000
posterior draws by keeping every 40th draw, producing ranks from 0 to 99.
Thinning is a precaution against autocorrelation, since uniform ranks assume
the retained draws are close to independent.

In [ ]:
weak = bayes.WEAK
ranks = []
for sim in range(200):
    # 1. Parameters from the prior
    alpha = rng.normal(weak["alpha_mean"], weak["alpha_sd"])
    beta = rng.normal(weak["beta_mean"], weak["beta_sd"])
    sigma = rng.exponential(1 / weak["sigma_rate"])

    # 2. Data from the model, with real predictors
    sample = hsls.sample(100, random_state=rng)[bayes.PREDICTORS].reset_index(drop=True)
    sample[bayes.OUTCOME] = rng.normal(alpha + sample.to_numpy() @ beta, sigma)

    # 3. Fit
    draws_sim = bayes.draws(bayes.sample(regression, bayes.stan_data(sample, weak), seed=sim + 1))
    kept = draws_sim.iloc[40::40]
    assert len(kept) == 99

    # 4. Rank of each true value among the kept draws
    true = {"alpha": alpha, "beta[1]": beta[0], "beta[2]": beta[1], "sigma": sigma}
    ranks.append({name: int((kept[name] < value).sum()) for name, value in true.items()})
ranks = pd.DataFrame(ranks)

In [ ]:
# With 200 simulations in 10 bins of 10 ranks each, a flat histogram has 20 per
# bin. The band is the central 99% range for one bin's count under a flat
# histogram, from the binomial distribution.
from scipy import stats

low, high = stats.binom.ppf([0.005, 0.995], len(ranks), 0.1)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for name, ax in zip(ranks.columns, axes):
    ax.axhspan(low, high, color="grey", alpha=0.25)
    ax.hist(ranks[name], bins=np.arange(0, 101, 10), edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("Rank of the true value (of 99 draws)")
axes[0].set_ylabel("Simulations")
fig.suptitle("Simulation-based calibration: 200 simulations, 100 students each")
fig.tight_layout()

counts = ranks.apply(lambda r: np.histogram(r, bins=np.arange(0, 101, 10))[0])
print(f"Band for one bin's count: {low:.0f} to {high:.0f} simulations")
print("Bins outside the band, by parameter:", ((counts < low) | (counts > high)).sum().to_dict())

In [ ]:
# Coverage: the share of simulations whose central 50% and 90% intervals hold
# the true value. Ranks give it directly: a central 90% interval of 99 draws
# runs from rank 5 to rank 94.
coverage = pd.DataFrame(
    {
        "50% interval": ranks.apply(lambda r: ((r >= 25) & (r <= 74)).mean()),
        "90% interval": ranks.apply(lambda r: ((r >= 5) & (r <= 94)).mean()),
    }
)
coverage.round(2)

Across the four ten-bin rank histograms, one or two bins straying outside the
99% uniform-variation band can happen by chance. Roughly flat histograms and
empirical coverage near 50% and 90% indicate no calibration failure at this
resolution. With 200 simulations and four one-parameter summaries, the check
catches posteriors that are clearly too narrow, too wide, or shifted, though
subtle errors remain beyond its reach. Calibration verifies only the
computation. Passing shows that the Stan program and sampler clear a test that
a coding or sampling error would likely fail, while leaving open whether the
model itself is right for HSLS:09.